In [3]:
import cupy as cp
from cupyx.scipy.linalg import lu, solve_triangular
import pandas as pd
import numpy as np

import time

In [13]:
def new_maxvol(A, r=None, delta=1e-3, max_iter=300):
    """
    Finds an approximate maximal-volume submatrix of A using CuPy for GPU acceleration.
    
    Parameters:
    -----------
    A : cupy.ndarray
        Input matrix of size (n, m).
    r : int, optional
        Size of the square submatrix to find. Defaults to min(n, m).
    delta : float, optional
        Convergence parameter. Defaults to 1e-3.
    max_iter : int, optional
        Maximum number of iterations. Defaults to 100.

    Returns:
    --------
    indices : list of int
        Indices of the rows forming the maximal-volume submatrix.
    """
    n, m = A.shape
    if r is None:
        r = min(n, m)
    
    # Initialize with the first r rows
    indices = cp.arange(r)
    A_sub = A[indices, :]
    
    # Compute initial inverse of A_sub
    A_sub_inv = cp.linalg.pinv(A_sub)
    B = cp.dot(A_sub_inv, A[indices, :])
    
    for _ in range(max_iter):
        # Find the largest entry in B in absolute value
        i, j = divmod(cp.argmax(cp.abs(B)), B.shape[1])
        if cp.abs(B[i, j]) <= 1 + delta:
            break  # Convergence reached
        
        # Swap rows in indices
        indices[i] = r + j
        
        # Update A_sub and its inverse using the Sherman-Morrison formula
        u = A[r + j, :] - A[indices[i], :]
        v = A_sub_inv[:, i]
        A_sub_inv -= cp.outer(cp.dot(A_sub_inv, u), v) / (1 + cp.dot(u, v))
        
        # Update B
        A_sub[i, :] = A[r + j, :]
        print(A_sub_inv.shape, A.shape)
        B = cp.dot(A_sub_inv, A)
    
    return indices.tolist()

In [14]:
def maxvol(A, e=1.05, k=100, test=False):
    """
    Compute the maximal-volume submatrix for given tall matrix using CuPy for GPU acceleration.

    Args:
        A (cp.ndarray): tall matrix of the shape [n, r] (n > r).
        e (float): accuracy parameter (should be >= 1).
        k (int): maximum number of iterations (should be >= 1).
        test (bool): If True, use slower but more stable pinv-based computations.

    Returns:
        (cp.ndarray, cp.ndarray): The row indices I and coefficient matrix B.
    """
    n, r = A.shape

    if n <= r:
        raise ValueError('Input matrix should be "tall"')

    # LU decomposition on GPU
    P, L, U = lu(A, check_finite=False)
    I = cp.argmax(P[:, :r], axis=0)  # Indices of rows in the maximal-volume submatrix

    if not test:
        # Using triangular solvers
        Q = solve_triangular(U, A.T, trans=1, check_finite=False)
        B = solve_triangular(L[:r, :], Q, trans=1, check_finite=False, unit_diagonal=True, lower=True).T
    else:
        # Using pseudo-inverse for stability
        Q = cp.dot(cp.linalg.pinv(U), A.T)
        B = cp.dot(cp.linalg.pinv(L[:r, :]), Q).T

    # Iterative refinement
    for _ in range(k):
        i, j = divmod(cp.abs(B).argmax(), r)
        if cp.abs(B[i, j]) <= e:
            break

        I[j] = i

        bj = B[:, j]
        bi = B[i, :].copy()
        bi[j] -= 1.

        B -= cp.outer(bj, bi / B[i, j])

    return I, B

def maxsumcolumns(A, r):
    """
    Selects the indices of the `r` columns with the largest sums in a matrix, using CuPy.

    Args:
        A (cp.ndarray): Input matrix of shape [n, m].
        r (int): Number of columns to select.

    Returns:
        cp.ndarray: Indices of the `r` columns with the largest sums, sorted in descending order.
    """
    # Compute the sum of each column
    columns_sums = A.sum(axis=0)
    
    # Get the indices of the top `r` columns by sum, sorted in descending order
    columns_index = cp.argsort(columns_sums)[-r:][::-1]

    return columns_index

def cur_decomposition(A, test=False, rank=0):
    """
    Выполняет CUR-разложение для матрицы A с использованием CuPy для ускорения на GPU.
    
    Параметры:
        A (cp.ndarray): Исходная матрица размера (m, n).
        test (bool): Если True, используются стабильные, но медленные вычисления.
        rank (int): Ранг разложения. Если 0, определяется автоматически.

    Возвращает:
        C (cp.ndarray): Подматрица столбцов из A размера (m, c).
        U (cp.ndarray): Матрица соединения размера (c, r).
        R (cp.ndarray): Подматрица строк из A размера (r, n).
    """
    # Определяем ранг матрицы, если не задан
    if rank == 0:
        rank = cp.linalg.matrix_rank(A)

    # Выбираем строки для матрицы R
    start_time_maxvol = time.time()
    #I, B = maxvol(A, test=test)
    I = new_maxvol(A)
    end_time_maxvol = time.time()
    print(f'maxvol выполнен за {end_time_maxvol-start_time_maxvol} сек.')

    I = I[:rank]
    R = A[I, :]  # Подматрица строк

    # Выбираем столбцы для матрицы C
    start_time_columns = time.time()
    column_indices = maxsumcolumns(A,rank)
    end_time_columns = time.time()
    print(f'maxsumcolumns выполнен за {end_time_columns-start_time_columns} сек.')
    C = A[:, column_indices]  # Подматрица столбцов

    # Строим W (взаимодействие строк и столбцов)
    W = A[cp.ix_(I, column_indices)]

    # Вычисляем матрицу U
    U = cp.linalg.pinv(W)  # Псевдообратная матрица

    return C, U, R

In [15]:
def percent_metric(A_true, A_approx):
    """
    Вычисляет процентное совпадение между истинной матрицей и её приближением, используя CuPy.
    
    Параметры:
        A_true (cp.ndarray): Исходная матрица.
        A_approx (cp.ndarray): Приближённая матрица.

    Возвращает:
        float: Процент совпадения между A_true и A_approx.
    """
    # Округляем приближенную матрицу
    A_approx = cp.round(A_approx, 0)
    
    # Вычисляем процент совпадения
    match_count = (A_true == A_approx).sum()
    total_elements = A_true.size  # Количество элементов в матрице
    return match_count / total_elements

def remove_zeros(A):
    """
    Удаляет строки и столбцы из матрицы numpy, сумма элементов которых равна нулю.
    
    Параметры:
        matrix (numpy.ndarray): Входная матрица.
    
    Возвращает:
        numpy.ndarray: Матрица без строк и столбцов с нулевой суммой.
    """
    # Вычисляем суммы строк и столбцов
    row_sums = np.sum(A, axis=1)
    col_sums = np.sum(A, axis=0)
    
    # Находим индексы строк и столбцов с ненулевой суммой
    non_zero_rows = row_sums != 0
    non_zero_cols = col_sums != 0
    
    # Возвращаем отфильтрованную матрицу
    return A[non_zero_rows][:, non_zero_cols]



In [7]:
%%time
data_pandas = pd.read_csv('UI_data.csv')
data_numpy = data_pandas.drop(['userId'],axis=1).to_numpy()
print(f'размер матрицы: {data_numpy.shape}')
data_numpy = remove_zeros(data_numpy)
print(f'размер матрицы: {data_numpy.shape}')

размер матрицы: (12697, 2851)
размер матрицы: (12697, 2851)
CPU times: total: 3.23 s
Wall time: 4.28 s


In [ ]:
print(f'ранк матрицы: {np.linalg.matrix_rank(data_numpy)}')

In [16]:
mempool = cp.get_default_memory_pool()
pinned_mempool = cp.get_default_pinned_memory_pool()

In [17]:
#data_numpy = np.round(data_numpy,0)
data = cp.cuda.alloc_pinned_memory(data_numpy.nbytes)
data = cp.asarray(data_numpy).astype(cp.float32)

In [18]:
print(data.nbytes)
print(mempool.used_bytes())
print(mempool.total_bytes())
print(pinned_mempool.n_free_blocks())

144796588
387168768
868780032
1


In [19]:
%%time
C,G,R = cur_decomposition(data, test=True, rank=2851//2)

approximation = C@G@R

print(f"C shape: {C.shape}")
print(f"G shape: {G.shape}")
print(f"R shape: {R.shape}")

print(f"approximation shape: {approximation.shape}")
print(f'Норма разности: {cp.linalg.norm(data-approximation)}')
print(f'Процент совпадений: {cp.round(percent_metric(data, approximation)*100,2)}%')

(2851, 2851) (12697, 2851)


ValueError: Axis dimension mismatch